# Bike Rental Prediction – Machine Learning Analysis

## Goals
This project analyzes a dataset of bike rentals and develops predictive models using:
- Exploratory Data Analysis (EDA)
- Ordinary Least Squares (OLS) regression on log-transformed target
- Ridge regression with hyperparameter tuning
- Cross-validation and learning curves
- A non-linear model (Random Forest or Neural Network)
- Final evaluation, discussion, limitations & future work

The objective is to create a clear, self-contained notebook demonstrating the full ML workflow.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

sns.set(style="whitegrid")

---

# 2. Data Loading and Cleaning

In this section, we load both datasets separately, inspect their structure, and perform initial preprocessing:

### Bike Counter Dataset (`train.parquet`)
- Bike rental counts at various counting stations
- Location data (latitude, longitude)
- Temporal information

### Weather Dataset (`external_data.csv`)
- Temperature, humidity, wind speed
- Precipitation, atmospheric pressure
- Weather conditions

For each dataset, we will:
- Check data types and structure
- Handle missing values
- Extract datetime features (hour, day, month, etc.)

The datasets will be merged later during **Feature Engineering** (Section 4).

## 2.1 Bike Counter Dataset

In [ ]:
# Load bike counter data
bike_data = pd.read_parquet(Path("data") / "train.parquet")
bike_data.head()

One may observe that the dataset countains two sets of bike_counts, one has an applied logarithmic scale to the bike count.
This could have been done in the past due to mutiple reasons: 
- Handle skewness: Traffic data is often characterized by hours with low to zero observations (nights, very cold, rainy weather) and hours with a high frequency (rush hour, sunnny, ...)
- reduce variance
- Mitigate the effect of events such as public holidays, sunny weather or public events on the forecasting model 

the logarithmic manipulation is applied by

$
\log(bike count + 1)
$

the + 1 is applied to make the calculation feasible for bike_counts == 0


In [ ]:
bike_data.info()

In [ ]:
bike_data.describe()

In [ ]:
print("Missing values in bike data")
print(bike_data.isna().sum())



**Observations:**
- No missing values in the bike counter dataset
- 496,827 hourly observations across 56 counters at 30 unique locations
- Target variable: `bike_count` (also available as `log_bike_count`)

In [ ]:
import holidays

# Define French holidays
FR_holidays = holidays.FR(years=range(2019, 2022))

bike_data["FR_holidays"] = bike_data["date"].dt.date.isin(FR_holidays).astype(int)
print(f"Number of rows marked as holidays: {bike_data['FR_holidays'].sum()}")

14688 data inputs are marked as holiday in the total dataset

### Extract Datetime Features

Extract temporal components from the `date` column for later analysis and modeling.

In [ ]:
def encode_dates(X):
    X = X.copy()  # Ensure we're working on a copy
    # Encode the date information
    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["weekday"] = X["date"].dt.weekday  # 0=Monday, 6=Sunday
    X["hour"] = X["date"].dt.hour

    # add weekend column
    X["weekend"] = X["weekday"].isin([5, 6]).astype(int)

    return X

# apply date encoding to the bike data
bike_data = encode_dates(bike_data)

# get timesteps from data
print(f'{bike_data["date"].iloc[1] - bike_data["date"].iloc[0]}')


In [ ]:
bike_data.head()

## 2.2 Weather Dataset

Load the external weather data containing meteorological observations.


In [ ]:
weather_data = pd.read_csv(Path("data") / "external_data.csv")
weather_data.head()


In [ ]:
print(f"Weather data shape: {weather_data.shape}")
print(weather_data.columns.tolist())


In [ ]:
weather_data.info()


As the timestamp are not formatted as datetime, we bringthe date to datetime format

In [ ]:
weather_data["date"] = pd.to_datetime(weather_data["date"], errors="coerce")

In [ ]:
# Check missing values in weather data
missing_pct = (weather_data.isna().sum() / len(weather_data) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': weather_data.isna().sum(),
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("Missing values in weather data (sorted by % missing):")
print(missing_df[missing_df['Missing Count'] > 0].head(20))


### Key Weather Features

Based on the dataset documentation, the most relevant features for bike rental prediction are:

| Column | Description | Unit |
|--------|-------------|------|
| `t` | Temperature | Kelvin |
| `u` | Humidity | % |
| `ff` | Wind speed | m/s |
| `dd` | Wind direction | degrees |
| `rr1` | Precipitation (1h) | mm |
| `pmer` | Sea-level pressure | Pa |


### Extract Datetime Features from Weather Data


In [ ]:
# apply date encoding to the weather data
weather_data = encode_dates(weather_data)

# get timestep from weather data
print(f'{weather_data["date"].iloc[1] - weather_data["date"].iloc[0]}')


the timestep of the weatherdata is 3 hours

In [ ]:
# Summary statistics for key weather features
key_weather_cols = ['t', 'u', 'ff', 'dd', 'rr1', 'pmer']
weather_data[key_weather_cols].describe()


## 2.3 Dataset Summary

Compare the date ranges to ensure temporal overlap between datasets.


In [ ]:

print("\n Bike Counter Data:")
print(f"   Rows: {len(bike_data):,}")
print(f"   Date range: {bike_data['date'].min()} to {bike_data['date'].max()}")
print(f"   Unique counters: {bike_data['counter_id'].nunique()}")
print(f"   Unique sites: {bike_data['site_id'].nunique()}")

print("\n Weather Data:")
print(f"   Rows: {len(weather_data):,}")
print(f"   Date range: {weather_data['date'].min()} to {weather_data['date'].max()}")
print(f"   Weather stations: {weather_data['numer_sta'].nunique()}")

print("\n Both datasets loaded and datetime features extracted.")
print("   Ready for EDA and Feature Engineering (merging in Section 4).")

# overlapping time range
# Print the overlapping time range between bike_data and weather_data

# Find the latest start date and earliest end date
overlap_start = max(bike_data['date'].min(), weather_data['date'].min())
overlap_end = min(bike_data['date'].max(), weather_data['date'].max())

print("\nOverlapping time range between bike data and weather data:")
print(f"   From: {overlap_start}")
print(f"   To:   {overlap_end}")



### 2.4 Merge Bike and Weather Dataset

Remove any duplicates from the the weather dataset

In [ ]:
# check for duplicates
print(weather_data.duplicated(subset="date").sum())

Bring the weather dataset to the same timestep as the bike dataset by linearly interpolating the values to match one hour timestep

In [ ]:
# Interpolate linearly to get from 3 hour data to 1 hour data
# Set date as index and resample to hourly
weather_data = weather_data.reset_index()
weather_data = weather_data.drop_duplicates(subset="date")
weather_data = weather_data.set_index('date')
weather_data = weather_data.resample('h').interpolate(method='linear')
weather_data = weather_data.reset_index()

# Check new datashape
print(f"Weather data shape after resampling: {weather_data.shape}")

Merge the data

In [ ]:
# Merge bike data with weather data using a left join
merged_data = pd.merge(bike_data, weather_data, on="date", how="left")

# Drop redundant date columns to avoid duplicates in the final dataset
merged_data = merged_data.loc[:, ~merged_data.columns.str.endswith(("_x", "_y"))]  # Change 1: Drop `_x` or `_y` suffix columns

In [ ]:
from sklearn.preprocessing import FunctionTransformer

date_encoder = FunctionTransformer(encode_dates, validate=False)
sample_encoded = date_encoder.fit_transform(merged_data[["date"]]).head()
sample_encoded

In [ ]:
# Reapply the _encode_dates function to extract date-related columns
merged_data = encode_dates(merged_data)

# Verify the new columns
print(merged_data[["date", "year", "month", "day", "weekday", "weekend"]].head())


In [ ]:
print(merged_data.shape)


---

# 3. Exploratory Data Analysis (EDA)

The purpose of EDA is to gain insights into the dataset, including:

### Distribution of rental counts
Understanding the shape of the target variable, including benefits of log transformation.

### Temporal patterns
- Hourly trends  
- Weekly patterns  
- Monthly/seasonal changes  

### Relationships with covariates
Visualizing how temperature, humidity, wind speed, and other weather features correlate with demand.

### Correlation analysis
Examining relationships between numerical features to identify potential collinearity or influence.

In [ ]:
bike_data.nunique(axis=0)

---

# 4. Feature Engineering

To improve predictive performance, we engineer additional features:

### Time-based features
- Hour of day  
- Weekday vs weekend  
- Month / season  

### Weather transformations
- Non-linear terms (e.g., temperature²)  
- Binary indicators (e.g., extreme weather)

### Optional future extensions
- Lag features (previous hour/day rentals)  
- Rolling averages  

These engineered features form the input for the machine learning models.

In [ ]:
## 4.1 Check merged data and handle missing values



In [ ]:
# Check for missing values after merge
missing_after_merge = merged_data.isna().sum()
missing_cols = missing_after_merge[missing_after_merge > 0]
print(f"Columns with missing values after merge: {len(missing_cols)}")
print(missing_cols.sort_values(ascending=False).head(15))


## 4.2 Temperature Conversion and Weather Transformations

Convert temperature from Kelvin to Celsius for better interpretability and create derived weather features.


In [ ]:
# Convert temperature from Kelvin to Celsius
merged_data['temp_celsius'] = merged_data['t'] - 273.15

# Create temperature squared (captures non-linear relationship)
merged_data['temp_squared'] = merged_data['temp_celsius'] ** 2

## 4.3 Cyclical Encoding for Time Features

Encode cyclical features (hour, weekday, month) using sine and cosine transformations. This preserves the cyclical nature of time (e.g., hour 23 is close to hour 0).

In [ ]:
# Cyclical encoding for hour (24-hour cycle)
merged_data['hour_sin'] = np.sin(2 * np.pi * merged_data['hour'] / 24)
merged_data['hour_cos'] = np.cos(2 * np.pi * merged_data['hour'] / 24)

# Cyclical encoding for weekday (7-day cycle)
merged_data['weekday_sin'] = np.sin(2 * np.pi * merged_data['weekday'] / 7)
merged_data['weekday_cos'] = np.cos(2 * np.pi * merged_data['weekday'] / 7)

# Cyclical encoding for month (12-month cycle)
merged_data['month_sin'] = np.sin(2 * np.pi * merged_data['month'] / 12)
merged_data['month_cos'] = np.cos(2 * np.pi * merged_data['month'] / 12)

print("Cyclical features created:")
merged_data[['hour', 'hour_sin', 'hour_cos', 'weekday', 'weekday_sin', 'weekday_cos']].head(10)


## 4.4 Season Feature

Create a season feature based on month.


In [ ]:
# Create season feature (meteorological seasons)
def get_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    elif month in [3, 4, 5]:
        return 1  # Spring
    elif month in [6, 7, 8]:
        return 2  # Summer
    else:
        return 3  # Autumn

merged_data['season'] = merged_data['month'].apply(get_season)

# Cyclical encoding for season (4-season cycle)
merged_data['season_sin'] = np.sin(2 * np.pi * merged_data['season'] / 4)
merged_data['season_cos'] = np.cos(2 * np.pi * merged_data['season'] / 4)

# Season distribution
print("Season distribution:")
print(merged_data['season'].value_counts().sort_index())
print("\n0=Winter, 1=Spring, 2=Summer, 3=Autumn")

# Show cyclical encoding
print("\nSeason cyclical encoding:")
print(merged_data[['season', 'season_sin', 'season_cos']].drop_duplicates().sort_values('season'))


## 4.5 Feature Selection and Final Dataset

Select the relevant features for modeling and prepare the feature matrix `X` and target `y`.


In [ ]:
# Define feature columns for modeling
feature_cols = [
    # Time features (cyclical)
    'hour_sin', 'hour_cos',
    'weekday_sin', 'weekday_cos', 
    'month_sin', 'month_cos',
    'season_sin', 'season_cos',
    # Time features (binary)
    'weekend', 'FR_holidays',
    # Weather features
    'temp_celsius', 'temp_squared',
    'u',  # humidity
    'ff',  # wind speed
    'rr1',  # precipitation
    'pmer',  # pressure
    # Location
    'latitude', 'longitude'
]

# Target variable
target_col = 'bike_count'

print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")


In [ ]:
# Check for missing values in selected features
missing_in_features = merged_data[feature_cols].isna().sum()
print("Missing values in feature columns:")
print(missing_in_features[missing_in_features > 0])

# Fill missing weather values with median (if any)
for col in feature_cols:
    if merged_data[col].isna().sum() > 0:
        merged_data[col] = merged_data[col].fillna(merged_data[col].median())
        print(f"Filled {col} with median")

print(f"\nTotal missing after filling: {merged_data[feature_cols].isna().sum().sum()}")


## 2.X Scale the data

In [ ]:


# Initialize the scaler
scaler = StandardScaler()

# Fit scaler on training data, but here we scale all for simplicity before split
merged_data[feature_cols] = scaler.fit_transform(merged_data[feature_cols])

print("Feature columns scaled using StandardScaler.")


In [ ]:
X = merged_data[feature_cols]
y = merged_data[target_col]


In [ ]:
# create train and validaiton set
cutoff_date = merged_data['date'].max() - pd.Timedelta("30 days")
mask = merged_data['date'] < cutoff_date
X_train, y_train = merged_data[mask][feature_cols], merged_data[mask][target_col]
X_val, y_val = merged_data[~mask][feature_cols], merged_data[~mask][target_col]

---

# 5. Baseline Model — Poisson Least Squares (OLS)

We train a simple Poisson Regression model using:

- Log-transformed rental counts (`log1p(count)`)
- Feature set engineered in Section 4  
- Evaluation on test data using RMSE and R²  

This acts as the baseline for comparing more advanced models.

### Interpretation
We analyze regression coefficients to understand which features increase or decrease expected rental demand.

In [ ]:
model = PoissonRegressor()
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

model.score(X_val, y_val)

In [ ]:
# Calculate metrics
from sklearn.metrics import mean_absolute_error

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("=" * 50)
print("LINEAR REGRESSION - Model Performance")
print("=" * 50)
print(f"R² Score:  {r2:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"MAE:       {mae:.4f}")


In [ ]:
# To color points by their value, use the `c` parameter and a colormap.
# For example, color by the true value (y_val) using a colormap like 'viridis':
plt.scatter(y_val, y_pred, c=y_val, cmap='viridis', alpha=0.7)
# If you want to color by predicted values instead, use c=y_pred.
plt.xlabel("Acutal bike counts")
plt.ylabel("predicted bike counts")
plt.title("Actual vs Predicted")
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', label='Perfect Prediction')
plt.legend()

plt.show()


In [ ]:
# Time series visualization: Actual vs Predicted for a sample counter
val_data = merged_data[~mask].copy()
val_data['y_pred'] = y_pred

# Pick one counter for visualization
sample_counter = val_data['counter_id'].iloc[0]
sample_data = val_data[val_data['counter_id'] == sample_counter].sort_values('date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sample_data['date'], sample_data['log_bike_count'], label='Actual', alpha=0.7)
ax.plot(sample_data['date'], sample_data['y_pred'], label='Predicted', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('log(bike_count)')
ax.set_title(f'Actual vs Predicted over Time (Counter: {sample_counter})')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


---

# 6. Regularized Model — Ridge Regression

Ridge Regression introduces L2 regularization to reduce overfitting in the linear model.

### Steps
- Standardize features using a pipeline  
- Tune the `alpha` parameter using GridSearchCV  
- Evaluate performance on the test set  
- Compare results with OLS  

Ridge often improves generalization and stabilizes coefficients.

In [ ]:
model = Ridge()
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

model.score(X_val, y_val)


In [ ]:
from sklearn.linear_model import Lasso
model = Lasso()
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

model.score(X_val, y_val)



---

# 7. Model Evaluation: Cross-Validation and Learning Curves

We evaluate the robustness and generalization of the models using:

### Cross-validation
- Compute cross-validated RMSE  
- Compare OLS and Ridge  

### Learning curves
Plot training and validation error as a function of training data size, allowing us to identify:

- High bias (underfitting)  
- High variance (overfitting)  
- Whether more data would help 


---

# 8. Advanced Model — Random Forest

We train a non-linear ensemble model capable of capturing complex interactions.

### Included steps
- Fit a Random Forest Regressor  
- Evaluate RMSE and R²  
- Analyze feature importances  

The Random Forest often provides strong predictive performance and helps reveal which features are truly influential.










 





---

# 9. Results and Discussion

In this section, we summarize and interpret the outcomes:

### Performance comparison
- OLS  
- Ridge Regression  
- Random Forest  

### Error patterns
Residual analysis to diagnose where models perform poorly (e.g., peak hours, extreme weather).

### Interpretation
Discuss why certain models performed better and what key factors drive bike rental demand.




---

# 10. Limitations and Future Work

### Limitations
- Dataset may not include all relevant drivers (events, holidays, bike availability).  
- Log-transform introduces mild bias when converting predictions back.  
- Standard regression models do not explicitly model temporal dependencies.  
- Random Forest lacks interpretability compared to linear models.  

### Future Work
- Add lagged features and rolling windows to capture temporal structure.  
- Explore time-series models (Prophet, ARIMA, LSTM).  
- Include richer weather and event-related datasets.  
- Investigate probabilistic forecasting for uncertainty quantification.  

---